# Token Merging Autoresearch Analysis

Systematic sweep over token merging strategies and ratios for ViT/VLM models.
Inspired by [karpathy/autoresearch](https://github.com/karpathy/autoresearch) --
run all experiments, track results in a TSV, and visualize the Pareto frontier.

In [ ]:
import subprocess, sys, os, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

## 1. Configuration

In [ ]:
STRATEGIES = ["bipartite", "kmeans", "average_pool"]
RATIOS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
MODELS = ["vit"]  # add "vlm" for CLIP
DEVICE = "cpu"     # set to "cuda" if available
RUNS = 10          # forward passes per config for timing
WARMUP = 3

RESULTS_FILE = "results_tome.tsv"

# Quality thresholds (from program.md)
MIN_COSINE_SIM = 0.95
MIN_PRED_MATCH = 0.90

## 2. Run the Experiment Grid

Calls `benchmark_tome.py` for each (model, strategy, ratio) combination
and logs results to the TSV.

In [ ]:
def parse_benchmark_output(output: str) -> dict:
    """Parse the structured output from benchmark_tome.py."""
    result = {}
    for line in output.strip().split("\n"):
        if ":" in line and not line.startswith("---"):
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            try:
                result[key] = float(val)
            except ValueError:
                result[key] = val
    return result


def run_single(model, strategy, ratio):
    """Run benchmark_tome.py and return parsed results."""
    cmd = [
        sys.executable, "benchmark_tome.py",
        "--model", model,
        "--strategy", strategy,
        "--ratio", str(ratio),
        "--device", DEVICE,
        "--runs", str(RUNS),
        "--warmup", str(WARMUP),
    ]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if proc.returncode != 0:
            print(f"  CRASH: {strategy} r={ratio}")
            return None
        return parse_benchmark_output(proc.stdout)
    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT: {strategy} r={ratio}")
        return None

In [ ]:
rows = []
total = len(MODELS) * len(STRATEGIES) * len(RATIOS)
i = 0

for model_name in MODELS:
    for strategy, ratio in product(STRATEGIES, RATIOS):
        i += 1
        print(f"[{i}/{total}] {model_name} | {strategy} | ratio={ratio} ... ", end="", flush=True)
        t0 = time.time()
        result = run_single(model_name, strategy, ratio)
        elapsed = time.time() - t0

        if result is None:
            rows.append({
                "model": model_name, "strategy": strategy, "ratio": ratio,
                "cosine_sim": 0.0, "speedup": 0.0, "pred_match": 0.0,
                "kl_div": 999.0, "base_ms": 0.0, "merged_ms": 0.0,
                "peak_mem_mb": 0.0, "score": 0.0, "status": "crash",
            })
            continue

        cos = result.get("cosine_sim", 0.0)
        spd = result.get("speedup", 0.0)
        pm = result.get("pred_match", 0.0)
        score = cos * spd

        meets_constraints = cos >= MIN_COSINE_SIM and pm >= MIN_PRED_MATCH

        row = {
            "model": model_name,
            "strategy": strategy,
            "ratio": ratio,
            "cosine_sim": cos,
            "speedup": spd,
            "pred_match": pm,
            "kl_div": result.get("kl_div", 0.0),
            "base_ms": result.get("base_ms", 0.0),
            "merged_ms": result.get("merged_ms", 0.0),
            "peak_mem_mb": result.get("peak_mem_mb", 0.0),
            "score": score,
            "status": "keep" if meets_constraints else "discard",
        }
        rows.append(row)
        print(f"score={score:.4f}  cos={cos:.4f}  spd={spd:.2f}x  ({elapsed:.1f}s)")

df = pd.DataFrame(rows)
df.to_csv(RESULTS_FILE, sep="\t", index=False)
print(f"\nSaved {len(df)} results to {RESULTS_FILE}")

## 3. Load Results

If you already ran the grid, just load from the TSV.

In [ ]:
df = pd.read_csv(RESULTS_FILE, sep="\t")
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["cosine_sim"] = pd.to_numeric(df["cosine_sim"], errors="coerce")
df["speedup"] = pd.to_numeric(df["speedup"], errors="coerce")

valid = df[df["status"] != "crash"].copy()
print(f"Total: {len(df)} experiments, {len(valid)} valid, {len(df) - len(valid)} crashed")
print(f"Strategies: {valid['strategy'].unique().tolist()}")
print(f"Ratios: {sorted(valid['ratio'].unique().tolist())}")
df.head(10)

## 4. Best Configurations

In [ ]:
kept = valid[valid["status"] == "keep"].copy()
kept_sorted = kept.sort_values("score", ascending=False)

print(f"Configs meeting quality thresholds (cos>={MIN_COSINE_SIM}, pred>={MIN_PRED_MATCH}):")
print(f"{'Rank':>4}  {'Strategy':<14} {'Ratio':>5}  {'Score':>7}  {'CosSim':>7}  {'Speedup':>7}  {'PredMatch':>9}  {'KL Div':>8}")
print("-" * 85)
for rank, (_, row) in enumerate(kept_sorted.iterrows(), 1):
    print(f"{rank:4d}  {row['strategy']:<14} {row['ratio']:5.2f}  {row['score']:7.4f}  "
          f"{row['cosine_sim']:7.4f}  {row['speedup']:6.2f}x  {row['pred_match']:9.4f}  {row['kl_div']:8.6f}")

if len(kept_sorted) > 0:
    best = kept_sorted.iloc[0]
    print(f"\n>>> BEST: {best['strategy']} at ratio={best['ratio']:.2f}  "
          f"(score={best['score']:.4f}, speedup={best['speedup']:.2f}x, cos_sim={best['cosine_sim']:.4f})")

## 5. Pareto Frontier: Speedup vs Quality

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = {"bipartite": "#2ecc71", "kmeans": "#3498db", "average_pool": "#e74c3c"}
markers = {"bipartite": "o", "kmeans": "s", "average_pool": "D"}

for strategy in valid["strategy"].unique():
    subset = valid[valid["strategy"] == strategy]
    ax.scatter(
        subset["speedup"], subset["cosine_sim"],
        c=colors.get(strategy, "gray"),
        marker=markers.get(strategy, "o"),
        s=80, label=strategy, edgecolors="black", linewidths=0.5,
        zorder=3,
    )
    # Label each point with its ratio
    for _, row in subset.iterrows():
        ax.annotate(f"{row['ratio']:.1f}",
                    (row["speedup"], row["cosine_sim"]),
                    textcoords="offset points", xytext=(5, 5),
                    fontsize=7, alpha=0.7)

# Pareto frontier
pareto = valid.sort_values("speedup")
frontier_x, frontier_y = [], []
best_cos = -1
for _, row in pareto.iterrows():
    if row["cosine_sim"] > best_cos:
        frontier_x.append(row["speedup"])
        frontier_y.append(row["cosine_sim"])
        best_cos = row["cosine_sim"]
# Reverse for correct Pareto direction (high speedup, high quality)
pareto_desc = valid.sort_values("speedup", ascending=False)
frontier_x2, frontier_y2 = [], []
best_cos2 = -1
for _, row in pareto_desc.iterrows():
    if row["cosine_sim"] > best_cos2:
        frontier_x2.append(row["speedup"])
        frontier_y2.append(row["cosine_sim"])
        best_cos2 = row["cosine_sim"]
frontier_x2.reverse()
frontier_y2.reverse()
ax.plot(frontier_x2, frontier_y2, "k--", alpha=0.3, linewidth=1.5, label="Pareto frontier")

# Threshold lines
ax.axhline(y=MIN_COSINE_SIM, color="red", linestyle=":", alpha=0.4, label=f"min cos_sim={MIN_COSINE_SIM}")

ax.set_xlabel("Speedup (x)", fontsize=12)
ax.set_ylabel("Cosine Similarity (higher = better quality)", fontsize=12)
ax.set_title("Token Merging: Speedup vs Output Quality", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("tome_pareto.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to tome_pareto.png")

## 6. Heatmaps: Score by Strategy x Ratio

In [ ]:
for model_name in df["model"].unique():
    model_df = valid[valid["model"] == model_name]
    pivot_score = model_df.pivot_table(index="strategy", columns="ratio", values="score")
    pivot_cos = model_df.pivot_table(index="strategy", columns="ratio", values="cosine_sim")
    pivot_spd = model_df.pivot_table(index="strategy", columns="ratio", values="speedup")

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for ax, pivot, title, cmap in [
        (axes[0], pivot_score, "Score (cos_sim * speedup)", "YlGn"),
        (axes[1], pivot_cos, "Cosine Similarity", "RdYlGn"),
        (axes[2], pivot_spd, "Speedup (x)", "YlOrRd"),
    ]:
        im = ax.imshow(pivot.values, cmap=cmap, aspect="auto")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{c:.1f}" for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_xlabel("Keep Ratio")
        ax.set_title(f"{title} ({model_name})")

        # Annotate cells
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8)

        plt.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.savefig(f"tome_heatmap_{model_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to tome_heatmap_{model_name}.png")

## 7. Latency Breakdown by Strategy

In [ ]:
fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(5 * len(STRATEGIES), 5), sharey=True)
if len(STRATEGIES) == 1:
    axes = [axes]

for ax, strategy in zip(axes, STRATEGIES):
    subset = valid[valid["strategy"] == strategy].sort_values("ratio")
    ax.bar(subset["ratio"].astype(str), subset["base_ms"],
           alpha=0.3, color="gray", label="Baseline")
    ax.bar(subset["ratio"].astype(str), subset["merged_ms"],
           alpha=0.8, color=colors.get(strategy, "blue"), label="Merged")
    ax.set_xlabel("Keep Ratio")
    ax.set_ylabel("Latency (ms)")
    ax.set_title(strategy)
    ax.legend(fontsize=8)

plt.suptitle("Forward Pass Latency: Baseline vs Merged", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("tome_latency.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Summary & Recommendation

In [ ]:
kept = valid[valid["status"] == "keep"].copy()

if len(kept) == 0:
    print("No configs met the quality thresholds!")
    print("Consider relaxing MIN_COSINE_SIM or MIN_PRED_MATCH.")
    print("\nTop 5 by score (regardless of threshold):")
    top = valid.nlargest(5, "score")
    for _, row in top.iterrows():
        print(f"  {row['strategy']:<14} r={row['ratio']:.2f}  "
              f"score={row['score']:.4f}  cos={row['cosine_sim']:.4f}  spd={row['speedup']:.2f}x")
else:
    best = kept.loc[kept["score"].idxmax()]
    print("=" * 60)
    print("  RECOMMENDED TOKEN MERGING CONFIGURATION")
    print("=" * 60)
    print(f"  Strategy:      {best['strategy']}")
    print(f"  Keep ratio:    {best['ratio']:.2f}")
    print(f"  Speedup:       {best['speedup']:.2f}x")
    print(f"  Cosine sim:    {best['cosine_sim']:.4f}")
    print(f"  Pred match:    {best['pred_match']:.4f}")
    print(f"  KL divergence: {best['kl_div']:.6f}")
    print(f"  Score:         {best['score']:.4f}")
    print("=" * 60)
    print(f"\nApply with:")
    print(f"  python main.py --model vit --apply-tome "
          f"--merge-strategy {best['strategy']} --merge-ratio {best['ratio']:.2f} --save-merged")

    # Per-strategy best
    print(f"\nBest per strategy:")
    for strategy in kept["strategy"].unique():
        s_best = kept[kept["strategy"] == strategy].nlargest(1, "score").iloc[0]
        print(f"  {strategy:<14} ratio={s_best['ratio']:.2f}  "
              f"score={s_best['score']:.4f}  speedup={s_best['speedup']:.2f}x  cos={s_best['cosine_sim']:.4f}")